# Neon isotopes: HFB fidelity and maximum Gaussian overlap

This notebook explains and runs the two Gaussian comparisons implemented by `study_gaussian_fidelity.py` for $^{20,22,24}$Ne with the USDB interaction.

The two questions are different:

1. **Energy-optimized HFB:** minimize $\langle\Phi|H|\Phi\rangle$ with the average neutron and proton numbers constrained, and then measure its fidelity with the exact ground state.
2. **Maximum-overlap Gaussian:** search directly over pure Bogoliubov vacua for the state that maximizes its overlap with the exact ground state.

Neither calculation imposes $M=0$ on the intrinsic Gaussian. The $M=0$ restriction is used only to reduce the exact-diagonalization basis.

## 1. What is being compared?

With an $^{16}$O core, Neon has two valence protons and

$$N_{\mathrm{val}}=A-18$$

valence neutrons. Therefore the three cases are

| Nucleus | Valence neutrons | Valence protons |
|---|---:|---:|
| $^{20}$Ne | 2 | 2 |
| $^{22}$Ne | 4 | 2 |
| $^{24}$Ne | 6 | 2 |

Exact diagonalization produces $|\Psi_0\rangle$ in the fixed $(N,Z,M=0)$ sector. The intrinsic HFB state $|\Phi_E\rangle$ is obtained from

$$|\Phi_E\rangle=\arg\min_{\Phi\in\mathcal G} \langle\Phi|H|\Phi\rangle,$$

subject to $\langle N\rangle=N_{\mathrm{val}}$ and $\langle Z\rangle=2$. The closest Gaussian is instead

$$|\Phi_F\rangle=\arg\max_{\Phi\in\mathcal G}|\langle\Psi_0|\Phi\rangle|^2,$$

where $\mathcal G$ is the chosen real or complex pure-Gaussian manifold. Thus $|\Phi_E\rangle$ and $|\Phi_F\rangle$ generally need not be the same state.

## 2. Fidelity used in this study

The fidelity used for both Gaussian states is the direct, unprojected overlap with the exact ground state:

$$F=|\langle\Psi_0|\Phi\rangle|^2.$$

No projection or renormalization is applied to $|\Phi\rangle$. The value therefore includes all number and angular-momentum symmetry breaking of the intrinsic Gaussian. Neither a target-sector weight nor the ratio obtained by dividing by that weight is included in this analysis; those belong to a projected-state study.

In [ ]:
from pathlib import Path
import json
import sys

# Find the repository whether the notebook starts in the root or a child folder.
ROOT = Path.cwd().resolve()
while not (ROOT / "study_gaussian_fidelity.py").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Run this notebook from inside the repository")
    ROOT = ROOT.parent

for directory in (ROOT, ROOT / "src" / "NSMFermions", ROOT / "benchmarks"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))

print("Repository:", ROOT)

## 3. Choose the calculation

The production calculation is intentionally disabled by default because exact diagonalization plus multiple non-convex optimizations can take a long time. Set `RUN_CALCULATION=True` when the environment is ready.

Start with only `MASSES=(20,)`. Once the output and convergence diagnostics look sensible, change it to `(20, 22, 24)`. More starts improve protection against local minima; they do not change the variational manifold.

In [ ]:
RUN_CALCULATION = False
MASSES = (20, 22, 24)
REAL_BOGOLIUBOV = True       # False doubles the coordinates and allows complex vacua
HFB_STARTS = 8
HFB_MAXITER = 500
GAUSSIAN_STARTS = 8
GAUSSIAN_MAXITER = 2000
SEED = 8
OUTPUT = ROOT / "results" / "usdb_ne_gaussian_fidelity.json"

print("Output file:", OUTPUT)

## 4. Run the study

For each isotope the following cell performs, in order:

1. constrained, multistart HFB energy minimization using the local $G^{20}$ gradient;
2. exact diagonalization in fixed $(N,Z,M=0)$;
3. evaluation of the intrinsic HFB fidelity;
4. multistart maximization of the intrinsic Gaussian overlap using the analytic local overlap gradient;
5. checkpointing of the JSON report.

The closest-Gaussian search is a separate optimization. It does not reuse the fact that the HFB state minimized the energy.

In [ ]:
from study_gaussian_fidelity import run_study

report = None
if RUN_CALCULATION:
    report = run_study(
        interaction_name="usdb",
        isotopes=MASSES,
        variational_method="hfb",
        starts=HFB_STARTS,
        maxiter=HFB_MAXITER,
        gaussian_starts=GAUSSIAN_STARTS,
        gaussian_maxiter=GAUSSIAN_MAXITER,
        real_bogoliubov=REAL_BOGOLIUBOV,
        seed=SEED,
        output=OUTPUT,
    )
else:
    print("Calculation skipped. Set RUN_CALCULATION=True to run it.")

### Equivalent terminal command

The same calculation can be run without Jupyter:

```bash
python -u study_gaussian_fidelity.py \
  --interaction USDB \
  --isotopes 20 22 24 \
  --variational-method hfb \
  --starts 8 --maxiter 500 \
  --gaussian-starts 8 --gaussian-maxiter 2000 \
  --output results/usdb_ne_gaussian_fidelity.json
```

Add `--complex-bogoliubov` for the full complex manifold. The real result is a lower bound to the maximum over complex Gaussian states.

## 5. Load an existing or newly generated report

In [ ]:
if report is None and OUTPUT.exists():
    report = json.loads(OUTPUT.read_text(encoding="utf-8"))

if report is None:
    print("No report found yet. Run the study locally or copy the Slurm result to:")
    print(OUTPUT)
else:
    print("Status:", report.get("status"))
    print("Nuclei available:", [row["nucleus"] for row in report["results"]])

In [ ]:
def number(value, digits=6):
    return "--" if value is None else f"{float(value):.{digits}g}"

if report is not None:
    header = (
        f"{'nucleus':<9} {'dim':>8} {'E exact':>12} {'E HFB':>12} "
        f"{'F HFB':>12} {'F max':>12} {'err E HFB':>12} {'err E max':>12}"
    )
    print(header)
    print("-" * len(header))
    for row in report["results"]:
        print(
            f"{row['nucleus']:<9} {row['exact_dimension']:>8d} "
            f"{number(row['exact_energy']):>12} {number(row['variational_energy']):>12} "
            f"{number(row.get('variational_ground_state_fidelity', row.get('variational_ground_state_fidelity_raw'))):>12} "
            f"{number(row.get('closest_gaussian_ground_state_fidelity', row.get('closest_gaussian_ground_state_fidelity_raw'))):>12} "
            f"{number(row['variational_energy_relative_error']):>12} "
            f"{number(row['closest_gaussian_energy_relative_error']):>12}"
        )

## 6. How to interpret the HFB result

The HFB optimizer minimizes energy, not fidelity. Its constrained local gradient is

$$G^{20}=H^{20}-\lambda_NN^{20}-\lambda_ZZ^{20}.$$

A trustworthy stationary solution requires both

$$\|G^{20}\|<\epsilon_G$$

and sufficiently small number residuals. The JSON fields to inspect are `variational_converged`, `variational_stationarity_error`, `variational_numbers`, `variational_pairing_norm`, and `variational_attempts`.

A small pairing norm means the HFB solution collapsed to an HF-like determinant. That is a physical result of the minimization, not automatically a failure. For fidelity evaluation, the script then uses the occupied natural orbitals as a numerically stable Slater-boundary representation; the JSON field `variational_fidelity_state_chart` records when this happened.

In [ ]:
if report is not None:
    for row in report["results"]:
        print(f"\n{row['nucleus']}")
        print("  HFB converged:          ", row["variational_converged"])
        print("  HFB stationarity:       ", row.get("variational_stationarity_error"))
        print("  <N>, <Z>:               ", row["variational_numbers"])
        print("  pairing norm:           ", row["variational_pairing_norm"])
        print("  closest G converged:    ", row["closest_gaussian_converged"])
        print("  closest G gradient norm:", row["closest_gaussian_gradient_norm"])
        attempts = row.get("closest_gaussian_attempts", [])
        if attempts:
            print("  closest G starts:       ", len(attempts))

## 7. How to interpret the maximum-overlap Gaussian

The closest-Gaussian optimizer maximizes

$$F(Z)=|\langle\Psi_0|\Phi(Z)\rangle|^2$$

directly. Its stationarity criterion is the norm of the analytic local overlap gradient, reported as `closest_gaussian_gradient_norm`. Since the problem is non-convex, `closest_gaussian_converged=True` only establishes stationarity for the best start found; repeating with more starts and different seeds checks whether the same maximum is recovered.

This optimizer does **not** minimize energy and does not impose exact particle number. Therefore `closest_gaussian_energy` is a diagnostic, not the objective, and its relative energy error need not improve when fidelity improves.

In [ ]:
if report is not None:
    for row in report["results"]:
        f_hfb = row.get("variational_ground_state_fidelity", row.get("variational_ground_state_fidelity_raw"))
        f_max = row.get("closest_gaussian_ground_state_fidelity", row.get("closest_gaussian_ground_state_fidelity_raw"))
        absolute = f_max - f_hfb
        relative = 100 * absolute / f_hfb if f_hfb > 0 else None
        print(
            f"{row['nucleus']}: maximum-overlap minus HFB = {absolute:+.8f}"
            + ("" if relative is None else f" ({relative:+.3f}% relative)")
        )

## 8. Visualize the comparison

The shared plotting function shows intrinsic fidelities, energies, relative energy errors, and the final optimization residuals. It does not use target-sector weights or sector-conditioned fidelities. Missing isotopes in a running checkpoint are simply omitted.

The closest-Gaussian energy is only a diagnostic: that state was optimized for overlap, not energy.

In [ ]:
from plot_study_results import plot_report

if report is not None:
    print("Neon isotope-chain summary")
    display(plot_report(report))
    for row in report["results"]:
        isotope_report = dict(report)
        isotope_report["results"] = [row]
        print(f"Individual analysis: {row['nucleus']}")
        display(plot_report(isotope_report))
else:
    print("No report is loaded yet.")

## 9. Raw Gaussian fidelity is not projected fidelity

The report produced here compares intrinsic Gaussian states with the exact state. It does not perform angular-momentum projection of the closest Gaussian. Rotational projection creates a coherent sum

$$P_{J=0}|\Phi\rangle=\frac{1}{8\pi^2}\int d\Omega\,R(\Omega)|\Phi\rangle,$$

which is generally non-Gaussian. A low intrinsic fidelity may therefore become high after $J=0$ projection. For example, the tested $^{20}$Ne HF state changed from approximately $0.1021$ intrinsic fidelity to $0.9231$ after exact-grid $J=0$ PAV.

To study the Euler-grid convergence, projected energy, $J$, and non-Gaussianity of the energy-optimized intrinsic state, run:

```bash
python -u study_projected_nongaussianity.py \
  --interaction USDB --mass 20 \
  --m-max 17 --j-max 6 \
  --intrinsic-method hfb --starts 8 --maxiter 500
```

Maximizing fidelity *after* projection is a different variation-after-projection objective. Do not identify it with the raw closest-Gaussian calculation in this notebook.

## 10. Run the systematic study with Slurm

From the repository root on MareNostrum:

```bash
mkdir -p logs
ISOTOPES="20 22 24" sbatch \
  --export=ALL,RUN_MODE=fidelity,INTERACTION=usdb,INTRINSIC_METHOD=hfb,STARTS=8,MAXITER=500,GAUSSIAN_STARTS=8,GAUSSIAN_MAXITER=2000 \
  slurm/run_nuclear_projection.sbatch
```

The job writes to a job-specific folder below `results/nuclear_projection/`. Copy the resulting JSON to the path stored in `OUTPUT`, or change `OUTPUT` above to point to the downloaded file.

## 11. Recommended production protocol

For each isotope:

1. Run the real calculation with at least 8 HFB and 8 overlap starts.
2. Require `variational_converged` and `closest_gaussian_converged` to be true.
3. Check that the HFB numbers equal the requested valence numbers within the configured tolerance.
4. Repeat with a second seed. The lowest HFB energy and largest Gaussian fidelity should reproduce.
5. Increase `GAUSSIAN_MAXITER` if the overlap-gradient norm remains above tolerance.
6. Run selected isotopes with `--complex-bogoliubov`; a higher result shows that the real restriction was active.
7. Report the intrinsic fidelity without dividing by a target-sector weight.
8. Treat every projected fidelity as a separate PAV or VAP observable.

A systematic table should record the seed, number of starts, iteration limits, real/complex choice, gradient norms, exact-sector dimension, intrinsic fidelity, and energy relative error.